## 🎯 Learning Objectives
By the end of this module, you will:
1. Understand the data pipeline: individual samples → batches → training
2. Implement Dataset abstraction and TensorDataset for tensor-based data
3. Build DataLoader with intelligent batching, shuffling, and memory-efficient iteration
4. Experience data pipeline performance characteristics firsthand
5. Create download functions for real computer vision datasets

**Why this matters:**
- **Learning:** Complete data loading system in one focused module for deep understanding
- **Production:** Proper organization like PyTorch's torch.utils.data with all core data utilities
- **Efficiency:** Optimized data pipelines are crucial for training speed and memory usage
- **Integration:** Works seamlessly with training loops to create complete ML systems

In [1]:
# Essential imports for data loading
import random
import sys
import time
from abc import ABC, abstractmethod
from typing import Iterator, List, Tuple

import numpy as np
rng = np.random.default_rng(7)

# Import real Tensor class from tinytorch package
from tinytorch.core.tensor import Tensor

## 💡 Understanding the Data Pipeline

Before we implement anything, let's understand what happens when neural networks "eat" data. The journey from raw data to trained models follows a specific pipeline that every ML engineer must master.


### The Data Pipeline Journey

Imagine you have 50,000 images of cats and dogs, and you want to train a neural network to classify them:

```
Raw Data Storage          Dataset Interface         DataLoader Batching         Model Input
┌─────────────────┐      ┌──────────────────┐      ┌────────────────────┐      ┌─────────────┐
│ cat_001.jpg     │      │ dataset[0]       │      │ Batch 1:           │      │ model(batch)│
│ dog_023.jpg     │ ───> │ dataset[1]       │ ───> │ [cat, dog, cat]    │ ───> │ compute     │
│ cat_045.jpg     │      │ dataset[2]       │      │ Batch 2:           │      │ loss        │
│ ...             │      │ ...              │      │ [dog, cat, dog]    │      │ repeat      │
│ (50,000 files)  │      │ dataset[49999]   │      │ ...                │      │             │
└─────────────────┘      └──────────────────┘      └────────────────────┘      └─────────────┘

### Why This Pipeline Matters

**Individual Access (Dataset)**: Neural networks can't process 50,000 files at once. We need a way to access one sample at a time: "Give me image #1,247".

**Batch Processing (DataLoader)**: GPUs are parallel machines - they're much faster processing 32 images simultaneously than 1 image 32 times.

**Memory Efficiency**: Loading all 50,000 images into memory would require ~150GB. Instead, we load only the current batch (~150MB).

**Training Variety**: Shuffling ensures the model sees different combinations each epoch, preventing memorization.

### The Dataset Abstraction

The Dataset class provides a uniform interface for accessing data, regardless of whether it's stored as files, in memory, in databases, or generated on-the-fly:

```
Dataset Interface
┌─────────────────────────────────────┐
│ __len__()     → "How many samples?" │
│ __getitem__(i) → "Give me sample i" │
└─────────────────────────────────────┘
          ↑                ↑
     Enables for     Enables indexing
    loops/iteration   dataset[index]
```

**Connection to systems**: This abstraction is crucial because it separates *how data is stored* from *how it's accessed*, enabling optimizations like caching, prefetching, and parallel loading.

### Dataset Base Class Implementation

First, we implement the abstract Dataset base class that defines the interface.

1. TODO: Implement the abstract Dataset base class

    APPROACH:
    1. Use ABC (Abstract Base Class) to define interface
    2. Mark methods as @abstractmethod to force implementation
    3. Provide clear docstrings for subclasses

In [2]:
class Dataset(ABC):
    """
    Abstract base class for datasets.
    """

    @abstractmethod
    def __len__(self) -> int:
        """
        Returns the number of samples in the dataset.
        """
        pass

    @abstractmethod
    def __getitem__(self, idx: int):
        """
        Returns a single sample from the dataset at the given index.
        """
        pass

In [3]:
class MyDataset(Dataset):
    def __len__(self): return 100
    def __getitem__(self, idx): return idx

dataset = MyDataset()
print(len(dataset))
print(dataset[42])

100
42


### 🧪 Unit Test: Dataset Abstract Base Class

This test validates our Dataset abstract base class is properly defined.

**What we're testing**: Abstract methods are enforced, concrete implementations work

**Why it matters**: Foundation interface for all dataset types

**Expected**: Cannot instantiate abstract Dataset, can instantiate concrete implementations

In [4]:
def test_unit_dataset():
    """🧪 Test Dataset abstract base class."""
    print("🧪 Unit Test: Dataset Abstract Base Class...")

    # Test that Dataset is properly abstract
    try:
        dataset = Dataset()
        assert False, "Should not be able to instantiate abstract Dataset"
    except TypeError:
        print("✅ Dataset is properly abstract")

    # Test concrete implementation
    class TestDataset(Dataset):
        def __init__(self, size):
            self.size = size

        def __len__(self):
            return self.size

        def __getitem__(self, idx):
            return f"item_{idx}"

    dataset = TestDataset(10)
    assert len(dataset) == 10
    assert dataset[0] == "item_0"
    assert dataset[9] == "item_9"

    print("✅ Dataset interface works correctly!")

if __name__ == "__main__":
    test_unit_dataset()

🧪 Unit Test: Dataset Abstract Base Class...
✅ Dataset is properly abstract
✅ Dataset interface works correctly!


## 🏗️ TensorDataset - When Data Lives in Memory

Now let's implement TensorDataset, the most common dataset type for when your data is already loaded into tensors. This is perfect for datasets like MNIST where you can fit everything in memory.

### Understanding TensorDataset Structure

TensorDataset takes multiple tensors and aligns them by their first dimension (the sample dimension):

### Why TensorDataset is Powerful

**Memory Locality**: All data is pre-loaded and stored contiguously in memory, enabling fast access patterns.

**Vectorized Operations**: Since everything is already tensors, no conversion overhead during training.

**Supervised Learning Perfect**: Naturally handles (features, labels) pairs, plus any additional metadata.

**Batch-Friendly**: When DataLoader needs a batch, it can slice multiple samples efficiently.

### Real-World Usage Patterns

```
# Computer Vision
images = Tensor(shape=(50000, 32, 32, 3))  # CIFAR-10 images
labels = Tensor(shape=(50000,))            # Class labels 0-9
dataset = TensorDataset(images, labels)

# Natural Language Processing
token_ids = Tensor(shape=(10000, 512))     # Tokenized sentences
labels = Tensor(shape=(10000,))            # Sentiment labels
dataset = TensorDataset(token_ids, labels)

# Time Series
sequences = Tensor(shape=(1000, 100, 5))   # 100 timesteps, 5 features
targets = Tensor(shape=(1000, 10))         # 10-step ahead prediction
dataset = TensorDataset(sequences, targets)
```

The key insight: TensorDataset transforms "arrays of data" into "a dataset that serves samples".

2. TODO: Implement TensorDataset for tensor-based data

    APPROACH:
    1. Store all input tensors
    2. Validate they have same first dimension (number of samples)
    3. Return tuple of tensor slices for each index

In [11]:
class TensorDataset(Dataset):
    """
    Dataset wrapping tensors for supervised learning.

    Each sample is a tuple of tensors from the same index across all input tensors.
    All tensors must have the same size in their first dimension.

    HINTS:
    - Use *tensors to accept variable number of tensor arguments
    - Check all tensors have same length in dimension 0
    - Return tuple of tensor[idx] for all tensors
    """

    def __init__(self, *tensors):
        """
        Create dataset from multiple tensors.

        Args:
            *tensors: Variable number of Tensor objects

        All tensors must have the same size in their first dimension.
        """
        ### BEGIN SOLUTION
        assert len(tensors) > 0, "Must provide at least one tensor"

        # Store all tensors
        self.tensors = tensors

        # Validate all tensors have same first dimension
        first_size = len(tensors[0].data)  # Size of first dimension
        for i, tensor in enumerate(tensors):
            if len(tensor.data) != first_size:
                raise ValueError(
                    f"Tensor size mismatch in TensorDataset\n"
                    f"  ❌ Tensor 0 has {first_size} samples, but Tensor {i} has {len(tensor.data)} samples\n"
                    f"  💡 All tensors must have the same size in their first dimension (the sample dimension)\n"
                    f"  🔧 Check your data: features.shape[0] should equal labels.shape[0]\n"
                    f"     Example fix: labels = labels[:{first_size}] or features = features[:{len(tensor.data)}]"
                )

    def __len__(self) -> int:
        """
        Return number of samples (size of first dimension).

        HINT: All tensors have same first dimension (validated in __init__)
        """
        
        return len(self.tensors[0].data)

    def __getitem__(self, idx: int) -> Tuple[Tensor, ...]:
        """
        Return tuple of tensor slices at given index.

        HINTS:
        - Check idx < len(self) to prevent out-of-bounds access
        - Use generator expression with tuple() for clean syntax
        """
    
        if idx >= len(self) or idx < 0:
            raise IndexError(f"Index {idx} out of range for dataset of size {len(self)}")

        # Return tuple of slices from all tensors
        return tuple(Tensor(tensor.data[idx]) for tensor in self.tensors)

In [12]:
features = Tensor([[1, 2], [3, 4], [5, 6]])  # 3 samples, 2 features each
labels = Tensor([0, 1, 0])                    # 3 labels

In [13]:
dataset = TensorDataset(features, labels)
print(len(dataset))  # 3
print(dataset[1])    # (Tensor([3, 4]), Tensor(1))

3
(Tensor(data = [3. 4.], shape=(2,)), Tensor(data = 1.0, shape=()))


TODO: Return the total number of samples in the dataset

APPROACH:

1. Access the first tensor from self.tensors

2. Return length of its data (first dimension size)

In [14]:
features = Tensor([[1, 2], [3, 4], [5, 6]])  # 3 samples
labels = Tensor([0, 1, 0])
dataset = TensorDataset(features, labels)
print(len(dataset))  # 3

3


TODO: Return the sample at the given index

APPROACH:

1. Validate index is within bounds
2. Extract data at index from each tensor
3. Wrap each slice in a Tensor and return as tuple


In [15]:
features = Tensor([[1, 2], [3, 4], [5, 6]])
labels = Tensor([0, 1, 0])
dataset = TensorDataset(features, labels)
sample = dataset[1] # Returns: (Tensor([3, 4]), Tensor(1))

In [16]:
sample

(Tensor(data = [3. 4.], shape=(2,)), Tensor(data = 1.0, shape=()))